In [1]:
import os
import sys
import math
import time
import wave
import shutil
import pickle
import threading
import subprocess
import urllib.request
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Evita errores en directorios de configuración de solo lectura
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_cache")
os.environ.setdefault("DEEPFACE_HOME", os.path.abspath("./.deepface_cache"))

import cv2
import joblib
import imageio.v2 as imageio
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt

In [2]:
FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

def ensure_face_landmarker_model(model_path: str = "face_landmarker.task") -> str:
    '''Descarga face_landmarker.task si no existe localmente.'''
    
    path = Path(model_path)
    if path.exists() and path.stat().st_size > 100_000:
        return str(path)
    
    print(f"Descargando modelo MediaPipe FaceLandmarker en {path}...")
    urllib.request.urlretrieve(FACE_LANDMARKER_URL, str(path))
    print(f"Modelo descargado ({path.stat().st_size / 1e6:.2f} MB).")
    
    return str(path)

In [3]:
# MEDIAPIPE LANDMARKS
RIGHT_EYE_IDXS = [33, 160, 158, 133, 153, 144]   # Ojo derecho del sujeto
LEFT_EYE_IDXS = [362, 385, 387, 263, 373, 380]   # Ojo izquierdo del sujeto
RIGHT_IRIS_IDX = 468
LEFT_IRIS_IDX = 473
MOUTH_CORNERS = (61, 291)
MOUTH_VERT_PAIRS = [(81, 178), (13, 14), (311, 402)]
# 6 puntos faciales para PnP: Nariz, Mentón, Ojo izq. ext, Ojo der. ext, Comisura izq., Comisura der.
PNP_LANDMARK_IDXS = [1, 152, 263, 33, 291, 61]
# Modelo antropométrico 3D canónico (+X derecha, +Y abajo, +Z profundidad)
MODEL_3D_POINTS = np.array([
    (0.0, 0.0, 0.0),          # Nariz (1)
    (0.0, 330.0, 65.0),       # Mentón (152)
    (225.0, -170.0, 135.0),   # Ojo izq. externo (263)
    (-225.0, -170.0, 135.0),  # Ojo der. externo (33)
    (150.0, 150.0, 125.0),    # Comisura izq. (291)
    (-150.0, 150.0, 125.0),   # Comisura der. (61)
], dtype=np.float64)
# Frustum holográfico 3D proyectado alrededor del rostro
VISOR_BOX_3D = np.array([
    (-320.0, -340.0, 190.0), (320.0, -340.0, 190.0), (320.0, 380.0, 190.0), (-320.0, 380.0, 190.0),
    (-270.0, -290.0, -110.0), (270.0, -290.0, -110.0), (270.0, 320.0, -110.0), (-270.0, 320.0, -110.0),
], dtype=np.float64)

In [4]:
def open_camera_and_options(camera_index: int, model_path: str):
    cap = cv2.VideoCapture(camera_index, cv2.CAP_V4L2)
    if not cap.isOpened():
        print(f"No se pudo abrir la cámara {camera_index}. Verifica la conexión o usa otro índice.")
        return None, None
    
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    # opts = mp.tasks.vision.FaceLandmarkerOptions(
    #     base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
    #     running_mode=mp.tasks.vision.RunningMode.VIDEO,
    #     num_faces=1,
    #     output_face_blendshapes=True,
    # )
    
    opts = None
    
    return cap, opts

In [5]:
from ironmanhudv2.biometry_engine.engine_config import BiometryConfig, ParamPose3D, ParamEAR, ParamMAR

EAR_PARAMS = ParamEAR(
    threshold=0.21,
    consec_frames_blink=2,
    micro_sleep_seconds=1.2,
    exp_moving_avg_alpha=0.2,
)

MAR_PARAMS = ParamMAR(
    threshold=0.55,
    yawn_seconds=1.0,
    exp_moving_avg_alpha=0.45,
)

POSE3D_PARAMS = ParamPose3D(
    pitch_nod_threshold=-18.0,
    pitch_up_threshold=25.0,
    yaw_look_away_threshold=28.0,
    pose_alert_seconds=1.0,
    exp_moving_avg_alpha=0.4
)

BIOMETRY_CONFIG = BiometryConfig(
    model_path=ensure_face_landmarker_model(),
    params_EAR=EAR_PARAMS,
    params_MAR=MAR_PARAMS,
    params_pose3D=POSE3D_PARAMS,
    alarm_cooldown_seconds=1.8,
    history_len=75
)

In [6]:
from ironmanhudv2.renderer import IronManHUDRenderer

def run_ironman_hud(camera_index = 0, record_gif_path = None, gif_seconds = 30.0):
    ensure_face_landmarker_model()
    cap, opts = open_camera_and_options(camera_index, "face_landmarker.task")
    if cap is None: #or opts is None:
        return

    # Rest of the function implementation
    ironman_hud_renderer = IronManHUDRenderer()
    ironman_hud_renderer.setup(BIOMETRY_CONFIG)

    print("Iniciando Ironman HUD. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("No se pudo leer el cuadro de la cámara.")
            break
        
        # Aquí iría el procesamiento del cuadro con MediaPipe y la visualización del HUD
        # ...
        hud_frame = ironman_hud_renderer.render(frame)
        cv2.imshow("Ironman HUD", hud_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    
run_ironman_hud()

W0000 00:00:1795284358.320397   32712 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795284358.331828   32717 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795284358.342548   32717 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Iniciando Ironman HUD. Presiona 'q' para salir.


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/skinnydevi/Development/computer-vision/P5/.venv/lib/python3.11/site-packages/cv2/qt/plugins"
